In [ ]:
#@title [Run] **Setup and Authenticate**

# Import libraries

import os
import pandas as pd
import google.colab.data_table as dt

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

!pip install PyPDF2
from PyPDF2 import PdfMerger

!pip install colorama
import colorama
from colorama import Fore, Style

# Authenticate and create the PyDrive client.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.9 MB/s eta 0:00:00


# **Configuration**

In [ ]:
TYPE = {
    "10": {
        "type": "Arrangements",
        "subtype": "Swing",
        "color": Fore.GREEN,
    },
    "11": {
        "type": "Arrangements",
        "subtype": "12 Bar",
        "color": Fore.BLUE,
    },
    "12": {
        "type": "Arrangements",
        "subtype": "Bluesy",
        "color": Fore.LIGHTBLUE_EX,
    },
    "20": {
        "type": "Instrumentals",
        "subtype": "Swing",
        "color": Fore.YELLOW,
    },
    "21": {
        "type": "Instrumentals",
        "subtype": "12 Bar",
        "color": Fore.MAGENTA,
    },
    "30": {
        "type": "Lead Sheet",
        "subtype": "Swing",
        "color": Fore.RED,
    },
    "31": {
        "type": "Lead Sheet",
        "subtype": "12 Bar",
        "color": Fore.LIGHTRED_EX,
    },
    "32": {
        "type": "Lead Sheet",
        "subtype": "Bluesy",
        "color": Fore.LIGHTMAGENTA_EX,
    },
}


PART_SELECTION = [
  'Vocals',
  'Clarinet',
  'Tenor Saxophone',
  'Electric Guitar',
  'Rhythm Guitar',
  'Drums',
  'Bass',
  'Concert',
  'Bb instrument',
  'Trumpet'
]

PARTNAMES = {
  'Vocals': {
      'raw': ['voice', 'vocals', 'piano'],
      'alt': ['Full Score', 'Lead Sheet C', 'Rhythm Section', 'Rhythm Guitar']
  },
  'Clarinet': {
      'raw': ['clarinet', 'clarinet in bb'],
      'alt': ['Lead Sheet Bb', 'Tenor Saxophone'],
  },
  'Tenor Saxophone': {
      'raw': ['tenor saxophone'],
      'alt': ['Clarinet', 'Lead Sheet Bb'],
  },
  'Electric Guitar': {
      'raw': ['jazz guitar', 'electric guitar'],
      'alt': ['Lead Sheet C', 'Vocals', 'Rhythm Section'],
  },
  'Rhythm Guitar': {
          'raw': ['rhythm guitar'],
          'alt': ['Rhythm Section', 'Lead Sheet C', 'Vocals', 'Acoustic Guitar'],
  },
  'Bass': {
      'raw': ['upright bass', 'string bass', 'rhythm section'],
      'alt': ['Rhythm Section', 'Lead Sheet C', 'Vocals',],
  },
  'Drums': {
      'raw': ['drum set', 'rhythm section'],
      'alt': ['Rhythm Section', 'Lead Sheet C', 'Vocals',],
  },
  'Rhythm Section': {
      'raw': ['rhythm section'],
  },
  'Concert': {
      'raw': ['concert', 'concert'],
      'alt': ['Electric Guitar', 'Lead Sheet C'],
  },
  'Bb instrument': {
      'raw': ['bb instrument'],
      'alt': ['Clarinet', 'Lead Sheet Bb'],
  },
  'Lead Sheet C': {
      'raw': ['concert', 'concert ', 'lead sheet'],
  },
  'Lead Sheet Bb': {
      'raw': ['bb instrument', 'bb instruments'],
  },
  'Trumpet': {
      'raw': ['trumpet bb', 'trumpet in bb'],
      'alt': ['Clarinet', 'Lead Sheet Bb', 'Tenor Saxophone'],
  }
}

INDEX_BLACKLIST = [
                   'ZZZZ',  # Unfinished
                   '1020',  # Honeysuckle Rose
                   ]

# **Processing**

In [ ]:
#@title [Run] **Load songbook** (~20s)

MASTER_DIR = "The Vintage Ties 2021" #@param {type: "string"}

ROOT = drive.ListFile({'q': 'mimeType = "application/vnd.google-apps.folder" and title = "%s"' % MASTER_DIR}).GetList()[0]['id']

def ParsePartName(name):
  parts = name.split(' - ')
  if len(parts) < 3:
    print ("Incorrect partname format: [%s]" % name)
    return {"ok": False}
  return {
      "ok": True,
      "index": parts[0],
      "title": parts[1],
      "part": parts[2].split('.')[0].lower()
  }

def ParseSongDirName(name):
  parts = name.split(' ')
  return parts[0], ' '.join(parts[1:])

def PopulateParts(songlist):
  for song in songlist.values():
    parts = drive.ListFile({'q': 'mimeType = "application/pdf" and "%s" in parents and trashed = false' % song["id"]}).GetList()
    song["parts"] = {}
    for part in parts:
      info = ParsePartName(part["title"])
      if info["ok"]:
        song["parts"][info["part"]] = part["id"]
  return songlist

def GetSongBook():
  songlist = drive.ListFile({'q': 'mimeType = "application/vnd.google-apps.folder" and "%s" in parents' % ROOT}).GetList()
  songbook = {}
  for song in songlist:
    index, title = ParseSongDirName(song["title"])
    if index.startswith("2"):
      print("Processing [%s %s]" % (index, title))
    if index in INDEX_BLACKLIST:
      continue
    if index == "2013" and title == "Rigamarole":
      continue
    songbook[index] = {
        "id": song["id"],
        "index": index,
        "type": TYPE[index[:2]]["type"],
        "subtype": TYPE[index[:2]]["subtype"],
        "color": TYPE[index[:2]]["color"],
        "title": title,
    }

  songbook = PopulateParts(songbook)
  return songbook

#@title [Run] **Make books by parts**



def PopulatePartsFromRaw(book):
  for index in book:
    song = book[index]
    song["book_parts"] = {}
    for part in PARTNAMES:
      for raw_part in PARTNAMES[part]["raw"]:
        if raw_part in song["parts"]:
          song["book_parts"][part] = {
              "original_part": raw_part,
              "id": song["parts"][raw_part]
          }
          break

def PopulateAlternativeParts(book, part_selection=PART_SELECTION):
  for index in book:
    song = book[index]
    for part in part_selection:
      if part in song["book_parts"]:
        continue
      for alt_part in PARTNAMES[part]["alt"]:
        if alt_part in song["book_parts"]:
          song["book_parts"][part] = song["book_parts"][alt_part]
          break
      if part not in song["book_parts"]:
        print ("Could not populate [%s] part for [%s %s]. Parts present: %s" % (
            part,
            song["index"],
            song["title"],
            list(song["book_parts"].keys())
            ))
#@markdown **regenerate** *has to be checked for the first run!*
regenerate = True #@param {type: "boolean"}

print_book = True #@param {type: "boolean"}

if regenerate:
  book = GetSongBook()
  PopulatePartsFromRaw(book)
  PopulateAlternativeParts(book)

if print_book:
  for i in sorted(book):
    song = book[i]
    print("%s%s. %s" % (song["color"], i, song["title"]))


Processing [2013 SK Groove]
Processing [2019 Close Your Eyes]
Processing [2018 Tea For Two]
Processing [2017 Hop, Skip, and Jump]
Processing [2016 Special Delivery Stomp]
Processing [2015 Bouncin Rhythm]
Processing [2104 Sent For You Yesterday]
Processing [2014 Royal Family]
Processing [2103 Uncle Haggart's Blues]
Processing [2102 That D-Minor Thing]
Processing [2008 It Feels Good]
Processing [2012 Black Market Stuff]
Processing [2007 Woke Up Clipped]
Processing [2009 Broadway Holdover]
Processing [2011 Summit Ridge Drive]
Processing [2010 Jive At Five]
Processing [2001 Till Tom Special]
Processing [2006 Tickle Toe]
Processing [2002 New Orleans Bump]
Processing [2005 Rock-A-Bye Basie]
Processing [2101 Benny's Bugle]
Processing [2004 A Smo-o-oth One]
Processing [2003 Flying Home]
Could not populate [Vocals] part for [1010#C I'm Confessin']. Parts present: []
Could not populate [Clarinet] part for [1010#C I'm Confessin']. Parts present: []
Could not populate [Tenor Saxophone] part for [1

In [ ]:
#@title [Optional] **Interactive Table of Content**

def MakeBookDataFrame(book):
  bookdata = {
      'type': [],
      'index': [],
      'title': [],
  }
  for part in PART_SELECTION:
    bookdata[part] = []

  for index in book:
    song = book[index]
    bookdata["type"].append("%s/%s" % (song["type"], song["subtype"]))
    bookdata["index"].append(song["index"])
    bookdata["title"].append(song["title"])
    for part in PART_SELECTION:
      if part not in bookdata:
        bookdata[part] = []
      partname = 'NA'
      if part in song["book_parts"]:
        partname = song["book_parts"][part]["original_part"]
      bookdata[part].append(partname)
  return pd.DataFrame(bookdata, index=bookdata["index"])

book_df = MakeBookDataFrame(book)
dt.DataTable(book_df, include_index=False, max_columns=100)

,type,index,title,Vocals,Clarinet,Tenor Saxophone,Electric Guitar,Rhythm Guitar,Drums,Bass,Concert,Bb instrument,Trumpet
2013,Instrumentals/Swing,2013,SK Groove,rhythm section,bb instruments,bb instruments,electric guitar,rhythm section,rhythm section,rhythm section,electric guitar,bb instruments,bb instruments
2019,Instrumentals/Swing,2019,Close Your Eyes,rhythm section,clarinet in bb,tenor saxophone,electric guitar,rhythm section,rhythm section,rhythm section,electric guitar,clarinet in bb,clarinet in bb
2018,Instrumentals/Swing,2018,Tea For Two,rhythm section,tenor saxophone,tenor saxophone,electric guitar,rhythm section,rhythm section,rhythm section,electric guitar,tenor saxophone,tenor saxophone
2017,Instrumentals/Swing,2017,"Hop, Skip, and Jump",rhythm section,clarinet in bb,clarinet in bb,electric guitar,rhythm section,rhythm section,rhythm section,electric guitar,clarinet in bb,clarinet in bb
2016,Instrumentals/Swing,2016,Special Delivery Stomp,rhythm section,clarinet in bb,clarinet in bb,electric guitar,rhythm section,rhythm section,rhythm section,electric guitar,clarinet in bb,clarinet in bb
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3016,Lead Sheet/Swing,3016,Baby Won't You Please Come Home,concert,bb instrument,bb instrument,concert,concert,concert,concert,concert,bb instrument,bb instrument
2003,Instrumentals/Swing,2003,Flying Home,rhythm section,clarinet in bb,tenor saxophone,electric guitar,rhythm section,rhythm section,rhythm section,electric guitar,clarinet in bb,clarinet in bb
3208,Lead Sheet/Bluesy,3208,Do You Duty,concert,clarinet in bb,clarinet in bb,concert,concert,concert,concert,concert,clarinet in bb,clarinet in bb
3017,Lead Sheet/Swing,3017,Big Butter and Egg Man,concert,bb instrument,bb instrument,concert,concert,concert,concert,concert,bb instrument,bb instrument


# Setlists

## Past setlists

In [ ]:
lampshade_1126 = ['2004', '3031', '2009', '3018#Eb', '2005', '3102', '2003', '1017', '2012', '1014', '1008', '2011', '2010', '1005', '3034', '2001', '1003#Cm', '3032', '2007', '1023#Am', '3035', '1013', '2101', '1010']

pioneer_farms_2_11_23 = ['2010', '1004', '3003', '1003#Cm', '3025', '1013', '2003', '3035', '3005', '1025', '2011', '1008', '1023', '3021', '1014', '2001', '3102', '1018', '2009', '1005', '2007', '1103', '3010', '3201', '1016', '2012', '3023', '1026', '3016', '2005', '3034', '2101', '3020', '3018', '3037', '3028']
pf_print = ['1026']
pf_reprint = ['1004', '1003#Cm', '1013', '1008', '1014', '1018', '1005', '1016']
vocal_print = ['3005', '3016', '3034', '3035', '3028', '3037']

swingjunction_42223 =['2004', '3031', '2012', '1003#Bbm', '2001', '3102', '1026', '3025', '3023', '2003', '2010', '1023', '2007', '3010', '2005', '3201', '2009', '3032', '2101', '3021']

fed_230623 = ['2010', '1025', '3031', '1003#Cm', '2012', '1027', '3201', '2001', '1102', '3025', '1103', '2009', '1005', '3029', '1014', '1018', '2011', '1008', '3023', '2007', '1024', '1007', '1023', '3205', '1026', '1028', '2008', '1017', '1013', '2101', '3020']
fed_print = ['1023', '1026', '1027', '1028', '2007', '2009', '2011']
fed_print_tenor = ['1017', '1025', '2010', '2012']
fed_vocals = ['1017', '1023', '1025', '1026', '3029', '3205']

acsr_0821 = ['2012', '1025', '3031', '1014', '1103#Bb', '3013#Eb', '2007', '1005', '1018', '3102', '1024', '1007', '1008', '1026', '2011', '1013', '3016', '2008', '1023#Am', '3201', '2101', '3025']
acsr_0821_prints = ['3013#Eb', '1023#Am']


alx_2023 = ['1023#Am', '1025', '1102', '1013', '3035', '1017', '3023', '1018', '1105', '2009', '1024', '1029', '3031', '1005', '2007', '3205', '2011', '3016', '1026', '3012', '1008', '1103#Bb', '3025']

cm_0923_print = ['1025', '1105', '3035', '1103#Bb', '1023#Am', '1029']

cm_0923 = ['2010', '1025', '1013', '1027', '1105', '1017', '3035', '1018', '3023', '2007', '1103#Bb', '1023#Am', '3038', '1005', '3102', '1029', '2011', '1028', '1026', '2009', '3031', '2101', '3018#Eb']



joes_wedding = ['2010', '1025', '1013', '1003#Cm', '1105', '1017', '3035', '2003', '3023', '2007', '1103#Bb', '1023#Am', '3038', '1005', '3102', '1029', '2011', '1014', '1026', '2009', '3031', '2101', '3018#Eb']
battle_ascr_23 = ['1025', '1013', '2011', '1026', '1017', '1103#Bb', '1005', '1027', '3036']
pioneer_farms_2_10_24 = ['1025', '3031', '1003#Cm', '2003', '1102', '1026', '2012', '1007', '1017', '1103#Bb', '1001', '1023#Am', '3035', '1005', '2011', '1008', '1018', '1105', '1014', '1024', '1029', '2010', '3205', '1013', '3029', '2007', '3201', '3023', '2101', '3025']

untied_070624 = ['2004', '3031', '2009', '3016', '2005', '1018', '2003', '1023#Am', '2102', '3034', '2011', '2103', '3023', '2001', '3005', '2007', '3037', '3039', '2012', '1026', '3028', '2101', '3037', '1010']
untied_070624_print = ['1026', '3039', '2102', '2103']

highball_071324 = ['1025', '3031', '1023#Am', '1008', '1102', '1026', '2003', '1017', '1103#Bb', '1001', '2102', '3035', '1005', '2011', '1018', '1105', '1014', '1024', '1029', '3205', '2005', '1013', '2007', '3201', '3023', '2101', '3025']
highball_071324_print = ['1103#Bb', '2102', '1026']

untied_080324 = ['2004', '3031', '2009', '3019', '2005', '3015', '2003', '1023#Am', '2102', '1010', '2011', '2103', '2001', '3005', '2007', '3037', '2014', '2012', '3028', '2101']

united_092124 = ['2103', '3031', '2009', '3040', '3037#Bb', '1023#Am', '3029', '2102', '3019', '2012', '2010', '3012', '2007', '3005', '1003#Bbm', '3015#Eb', '3102', '3028', '3023', '3018', '3010', '3001', '3025']

untied_110224 = ['2004', '3031', '2009', '3016', '2014', '3009', '2003', '1023#Am', '2102', '3019', '2103', '3023', '2001', '3005', '2012', '3039', '2010', '3015', '2101', '3018', '3010', '3037']

vt_alx_2024 = ['1001', '1023#Am', '3031', '1013', '2011', '1017', '3016', '1026', '1105', '2102', '1024', '1025', '1027', '2007', '1005', '3213', '2005', '1029', '3205', '1018', '1008', '1103#Bb', '3025']

united_120724 = ['2103', '3031', '2009', '1027', '1023#Am', '3032', '2003', '3042', '2102', '3019', '2004', '3041', '2001', '3037', '2012', '3039', '2010', '3028', '2101', '3015', '3009', '3005']

untied_tol_2024 = ['1023#Am', '3031', '2012', '3039', '2010', '3005', '2102', '1027', '2101', '3019', '3037']

untied_01042025 = ['2103', '3031', '2009', '3016', '2005', '3034', '2003', '1023#Am', '2102', '3019', '2004', '3042', '2001', '3005', '2011', '3039', '2010', '3028', '2101', '3018', '3010', '3037']

hard_knox_2025 = ['2103', '3023', '1023#Am', '3010', '2012', '2009', '1007#LS', '2011', '2010', '2004', '3005', '2102', '3009#Ab', '1003', '2003', '3018', '1103#Bb', '2104', '1033', '2007', '3021#G', '1013', '2005', '3040#Eb', '2101', '3025']
hard_knox_2025_vocals = ['3023', '3010', '1007#LS', '3005', '3009#Ab', '1003', '3018', '1103#Bb', '1033', '3021#G', '1013', '3040#Eb', '3025']
vtq_batch_0425 = ['2103', '3009', '2009', '3005', '2012', '3042', '2001', '3031', '1023#Am', '2004', '3037', '2102', '3021#G', '2005', '3039', '2011', '3028', '2101', '3010', '2010']
vt_highball_51025_new = ['2015', '1030']
vtq_batch_0525 = ['2103', '3037', '2009', '3023', '2104', '3042', '2102', '3031', '1023#Am', '3103', '3016', '2012', '3034', '2007', '3039', '2011', '3028', '2101', '3010', '2010']
vt_highball_51025 = ['1025', '1008', '1013', '2011', '1030', '1026', '3035', '2102', '1024', '1029', '3031', '2015', '1105', '1005', '3213', '2005', '1018', '1103#Bb', '3025', '1102', '2012']
vtq_batch_0607 = ['2103', '3037', '2009', '3019', '3032', '3010', '2102', '3031', '2015', '2104', '3015', '2012', '3034', '2007', '3021', '3005', '3028', '2005']
vtq_batch_0705 = ['2103', '3037', '2009', '3047', '2012', '3005', '2004', '3046', '2011', '3045', '1023#Am', '2005', '3031', '2102', '3048', '2001', '3028', '2101', '3010', '2010', '3019', '3032']
vtq_batch_0705_new = ['3045', '3046', '3047', '3048']

vtjb_highball_08092025 = ['1025', '1008', '1013', '2011', '1003#Gm', '1030', '1026', '3035', '2102', '1024', '1029', '3031', '2015', '1105', '1014', '1005', '3213', '2005', '1018', '1103#Bb', '3025', '1102', '2012']
vtjb_highball_08312025 = ['1025', '1014', '1030', '1013', '3035', '1026', '2011', '1008', '1024', '1029', '1027', '1017', '1105', '2015', '1005', '1102', '1018', '1103#Bb', '3025', '2012', '3031']

vtq_batch_0925 = ['2004', '3032', '2009', '3037', '2011', '3029', '2010', '3039', '2003', '1023#Am', '3023', '2102', '3042', '2009', '3031', '2012', '2001', '2101', '2103', '3031']

vtq_st_elmo_0925 = ['2103', '3023', '2009', '3010', '1023#Am', '3042', '2102', '3031', '2104', '3016', '2012', '3005', '2007', '3039', '2011', '3028', '2101', '3032', '2010']

vt_highball_09262025 = ['1001', '1014', '2003', '1025', '1013', '3035', '1018', '2011', '1005', '1029', '1030', '3023', '1105', '1026', '1017', '1008', '2015', '1024', '1103#Bb', '3031', '2102']
vtq_batch_10042025 = ['2004', '3031', '2009', '3023', '1023#Am', '3042', '2012', '3037', '2102', '2104', '3016', '2010', '3005', '2007', '3039', '2011', '3028', '2101', '3032', '2003']
vtq_bright_10112025 = ['2104', '3031', '2009', '3016', '1023#Am', '3042', '2012', '1027', '2003', '2102', '3023', '2010', '3032', '2007', '3039', '2011', '3028', '2101', '3029', '2001']
vt_highball_10122025 = ['1001', '1102', '1025', '2015', '3035', '1013', '2012', '1018', '2011', '1024', '1029', '1030', '1027', '1105', '1026', '3213', '1008', '2005', '1103#Bb', '3025', '3031', '2102']
vtq_acsr_10202025 = ['2004', '3031', '2009', '3005', '1023#Am', '3014', '2012', '1027', '2003', '2103', '3049', '2005', '3037', '3032', '2010', '3029', '2102', '3028', '2101', '3018', '2001']
vtq_acsr_10202025_print = ['3014', '1027', '3018', '3049']


vtq_nac_11082025 = ['2103', '3037', '2012', '3019', '3032', '3010', '2102', '3031', '3001', '2010', '2104', '3021', '2009', '3018', '3034', '2007', '3015#Eb', '3005', '3028', '2005']
vtjb_hou_11092025 = ['1001', '1102', '1025', '2015', '3035', '1013', '2012', '1018', '2011', '1024', '1029', '3031', '2102', '1105', '1026', '3213', '1008', '2005', '1103#Bb', '3025', '3023', '2003', '2101']
vtjb_alx_2025 = ['1029', '1025', '1102', '1030', '1013', '2011', '1001', '2015', '3035', '1024', '1023#Am', '1027', '2102', '1008', '1026', '1105', '1018', '2007', '1103#Bb', '3031', '2003', '3025']

vtjb_highball_12_2025 = ['1001', '1102', '1025', '2015', '3035', '1013', '2012', '1018', '2011', '1024', '1029', '1008', '2102', '1105', '1026', '1030', '3023', '2005', '1103#Bb', '3025', '3031', '2003', '2101']

vtq_batch_012026 = ['2103', '3037', '2012', '3032', '1003', '2004', '3042', '1023', '1032', '2009', '3041', '3039', '2003', '3015#Eb', '2101']

vtq_batch_012026_print = ['1023', '1032', "3015#Eb"]

vt_highball_01112026 = ['1001', '3031', '1014', '1025', '2005', '1018', '2011', '1005', '1029', '1008', '2015', '1030', '1013', '1105', '1026', '1024', '1102', '3023', '2003', '1017', '1103#Bb', '3025']

vtq_hrh_2026_new = ['2013', '2016', '2017', '2019', '2018']

vt_highball_02082026 = ['1001', '3031', '1102', '1025', '2005', '1017', '2010', '1005', '1023#Am', '1008', '2015', '3023', '1013', '3041', '1026', '1103#Bb', '3025']

vtq_hrh_2026 = ['2104', '3005', '2017', '3037', '2019', '3032', '2102', '3016', '2016', '2013', '3039', '2011', '3029', '2009', '3015#Eb', '2018', '2007', '2101', '2103', '2012', '2005', '3028']

vtq_batch_0326 = ['2004', '3031', '2017', '3037', '2012', '3032', '2102', '1027', '2016', '2104', '3039', '2011', '3016', '2009', '3015#Eb', '2018', '2007', '3010', '2010', '3028', '3045', '2003']

vtjb_highball_0326 = ['1001', '3031', '1102', '1005', '1105', '2015', '1025', '1023#Am', '1029', '1008', '1018', '1030', '1013', '2005', '1017', '1024', '3025']


## Current/Recent Setlists

In [ ]:
vtq_batch_0426 = ['2004', '3037', '2017', '3009', '2012', '3023', '2102', '3029', '3005', '2010', '2103', '3031', '2011', '3016', '2009', '3015#Eb', '2001', '3010', '3028', '2101', '3018', '3034', '2003']

vtjb_highball_042026 = ['1025', '1014', '3035', '1013', '3023', '2015', '1105', '1026', '1029', '3036', '1018', '2011', '3034', '1017', '1005', '2007', '1023#Am']

# Uploading

In [ ]:
#@title **Make a collection**
#@markdown The folders are never recreated, just the pdf files are updated.
def ClearDir(folder):
  query = [
       'mimeType = "application/pdf"',
       '"%s" in parents' % folder["id"]
  ]
  query = ' and '.join(query)
  filelist = drive.ListFile({'q': query}).GetList()
  if len(filelist) > 0:
    print("Deleting files in [%s]..." % folder["title"])
  for f in filelist:
    f.Delete()


def MakeDir(title, parent=None):
  dir = drive.CreateFile({
      'mimeType': "application/vnd.google-apps.folder",
      'title': title})
  if parent:
    dir["parents"] = [{'id': parent["id"]}]
  dir.Upload()
  parentname = ""
  if parent:
    parentname = parent["title"]
  print("> Making a new [%s/%s] folder." % (parentname, title))
  return dir

def CheckDir(title, parent=None):
  query = [
       'mimeType = "application/vnd.google-apps.folder"',
       'title = "%s"' % title,
       'trashed = False',
  ]
  parentname = ""
  if parent:
    query.append('"%s" in parents' % parent["id"])
    parentname = parent["title"]

  query = ' and '.join(query)
  dir_list = drive.ListFile({'q': query}).GetList()

  if (len(dir_list) == 0):
      print("Folder [%s/%s] does not exist." % (parentname, title))
      return
  print("Folder [%s/%s] found." % (parentname, title))

  return dir_list[0]

def FindOrMakeDir(title, parent=None):
  folder = CheckDir(title, parent)
  if not folder:
    folder = MakeDir(title, parent)
  return folder


def MakeCollectionFolders(
    top_folder_name, partnames, collection_name, is_gig, do_copy):
  top_folder = FindOrMakeDir(top_folder_name)

  part_folders = {}
  if (is_gig):
    gigs_folder = FindOrMakeDir("Setlists", top_folder)
    collection_folder = FindOrMakeDir(collection_name, gigs_folder)
  else:
    collection_folder = FindOrMakeDir(collection_name, top_folder)

  for part in partnames:
    part_folders[part] = FindOrMakeDir(part, collection_folder)
    if (do_copy):
      ClearDir(part_folders[part])

  return collection_folder, part_folders

def PrintCollection(songlist):
  for index in songlist:
    song = book[index]
    print("%s%s. %s" % (song["color"], index, song["title"]))
  print(Fore.RESET)

def SortPdfs(pdfs, order=None):
  if not order:
    return sorted(pdfs, key=(lambda x: x["title"]))

  order_map = {}
  for i in range(len(order)):
    order_map[order[i]] = i
  return sorted(pdfs, key=(lambda x: order_map[x["title"].split(' ')[0]]))

def MergeCollection(collection, collection_folder, part_folders, order=None):
  !mkdir downloads/
  merged_files = []
  for part in part_folders:
    print("Merging pdfs for [%s]..." % part)
    query = [
      'mimeType = "application/pdf"',
      '"%s" in parents' % part_folders[part]["id"],
      'trashed = False'
    ]
    query = ' and '.join(query)
    pdfs = drive.ListFile({'q': query}).GetList()
    merger = PdfMerger()
    for pdf in SortPdfs(pdfs, order):
      pdf.GetContentFile("downloads/%s" % pdf["title"])
      merger.append("downloads/%s" % pdf["title"])
    merged_files.append("downloads/merged_%s.pdf" % part.replace(' ', '_'))
    merger.write(merged_files[-1])
    merger.close()

    merged_file = drive.CreateFile({
        "title": "ALL - %s - %s.pdf" % (collection, part),
        "parents": [{"id": part_folders[part]["id"]}]}
    )
    merged_file.SetContentFile(merged_files[-1])
    merged_file.Upload()

  # Merge all
  merger = PdfMerger()
  for merged_file in merged_files:
    merger.append(merged_file)
  merger.write("downloads/full_merge_%s" % collection_name)
  merger.close()

  full_merge = drive.CreateFile({
      "title": "ALL PARTS - %s.pdf" % collection,
      "parents": [{"id": collection_folder["id"]}]}
  )
  full_merge.SetContentFile("downloads/full_merge_%s" % collection_name)
  full_merge.Upload()

  !rm -R downloads/
  print("Merging completed and uploaded.")

def CreateCollection(top_folder, parts, collection, songlist,
                     do_copy, do_merge, is_gig, order_by_index):
  if (order_by_index):
    songlist = sorted(songlist)

  PrintCollection(songlist)
  collection_folder, part_folders = MakeCollectionFolders(
      top_folder, parts, collection, is_gig, do_copy)

  if (do_copy):
    for index in songlist:
      song = book[index]
      print("Copying parts for [%s %s]..." % (song["index"], song["title"]))
      for part in part_folders:
        if part not in song["book_parts"]:
          print("[%s] chart is missing for [%s]" % (part, song["title"]))
          continue
        drive.auth.service.files().copy(
            fileId=song["book_parts"][part]["id"],
            body={"parents": [part_folders[part]]}).execute()
    print()
  if (do_merge):
    if (order_by_index):
      MergeCollection(collection, collection_folder, part_folders)
    else:
      MergeCollection(collection, collection_folder, part_folders, songlist)

top_folder = "Austin (admin)" #@param {type: "string"}
collection_name = "VT - Highball - April 2026" #@param {type: "string"}
songlist =   vtjb_highball_042026#@param {type: "raw"}

# parts = PART_SELECTION
parts =  [ 'Clarinet', 'Electric Guitar', 'Rhythm Guitar', 'Bass', 'Tenor Saxophone', 'Drums', 'Vocals']#@param {type: "raw"}

copy = True #@param {type: "boolean"}
merge = True #@param {type: "boolean"}
gig_collection = True #@param {type: "boolean"}
order_by_index = False #@param {type: "boolean"}


CreateCollection(top_folder,
                 parts,
                 collection_name,
                 songlist,
                 copy, merge, gig_collection, order_by_index)


1025. Spreadin' Rhythm Around
1014. There ain't no sweet man
3035. Sentimental Journey
1013. Swing Brother Swing
3023. Sugar
2015. Bouncin Rhythm
1105. Boogie Blues
1026. Swingin On Nothin
1029. Evenin
3036. I Can't Give You Anything But Love
1018. It Ain't Right
2011. Summit Ridge Drive
3034. How High The Moon
1017. Sweet Lotus Blossom
1005. Honey, It must be Love
2007. Woke Up Clipped
1023#Am. No Moon At All

Folder [/Austin (admin)] found.
Folder [Austin (admin)/Setlists] found.
Folder [Setlists/VT - Highball - April 2026] found.
Folder [VT - Highball - April 2026/Clarinet] found.
Deleting files in [Clarinet]...
Folder [VT - Highball - April 2026/Electric Guitar] found.
Deleting files in [Electric Guitar]...
Folder [VT - Highball - April 2026/Rhythm Guitar] found.
Deleting files in [Rhythm Guitar]...
Folder [VT - Highball - April 2026/Bass] found.
Deleting files in [Bass]...
Folder [VT - Highball - April 2026/Tenor Saxophone] found.
Deleting files in [Tenor Saxophone]...
Folder [VT 

In [ ]:
 !rm -R downloads/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Standard collections

In [ ]:
CreateCollection("TourLS", ["Rhythm Guitar", "Clarinet"], "LS", collections['Lead Sheets'], True, True, True, True)

3001. The Mood that I'm In
3002. All of Me
3003. Can't Help Lovin Dat Man
3004. My Heart Belongs to Daddy
3005. Back in Your Own Backyard
3006. Dream A Little Dream Of Me
3007. The Man I Love
3008. Everybody Loves My Baby
3009. Keeping out of Mischief Now
3009#Eb. Keeping out of Mischief Now
3010. I Can't Believe that You're In Love with Me
3011. Bill Bailey
3012. Lonesome Road
3013. Why
3013#Eb. Why
3014. You're the Cream in My Coffee
3015. Ain't Misbehavin'
3016. Baby Won't You Please Come Home
3017. Big Butter and Egg Man
3018. Exactly Like You
3019. Home (When Shadows Fall)
3020. I Only Have Eyes For You
3021. If I Could Be With You
3022. I'm Just a Lucky So and So
3023. Sugar
3024. Trav'lin All Alone
3025. No Love, No Nothin
3101. Fujiyama Mama
3102. Fine and Mellow
3201. Send Me to the Electric Chair
3202. Why Don't You Do Right
3203. Put the Blame on Mame
3204. I Ain't Got Nothin (But the Blues)
3205. Kitchen man
3206. Am I Blue
3208. Do You Duty
3209. Mama Goes Where Papa Goes


In [ ]:
collections = {
    "All charts": list(book.keys()),
    "Arrangements": [s for s in book.keys() if TYPE[s[:2]]["type"] == "Arrangements"],
    "Instrumentals": [s for s in book.keys() if TYPE[s[:2]]["type"] == "Instrumentals"],
    "Lead Sheets": [s for s in book.keys() if TYPE[s[:2]]["type"] == "Lead Sheet"]
}
for name in collections:
  collection = collections[name]
  CreateCollection("Vintage Ties 2021 Managed - Example 2", PART_SELECTION, name, collection,
                   do_copy=True, do_merge=True, is_gig=False, order_by_index=True)

KeyboardInterrupt: ignored

In [ ]:
austin_tier_1 = [
  '1004','1005','1006','1007','1008','1010','1011','1013','1014','1015','1016','1018','1102','1103','3001','3003','3005','3009','3010','3013','3019','3020','3021','3022','3024','3025','3102','3205'
]

In [ ]:
CreateCollection("Vintage Ties 2021 Managed - Example 2", PART_SELECTION, "Austin Tier 1", austin_tier_1,
                   do_copy=True, do_merge=True, is_gig=True, order_by_index=True)

# Scraptchpad

# Interface experiments

In [ ]:
import IPython
import os
IPython.display.HTML(filename='/path/to/your/filename')

In [ ]:

f = open("dragndrop/dnd.html", "a")
f.write("""
<div cdkDropListGroup>
  <div class="example-container">
    <h2>To do</h2>

    <div
      cdkDropList
      [cdkDropListData]="todo"
      class="example-list"
      (cdkDropListDropped)="drop($event)">
      <div class="example-box" *ngFor="let item of todo" cdkDrag>{{item}}</div>
    </div>
  </div>

  <div class="example-container">
    <h2>Done</h2>

    <div
      cdkDropList
      [cdkDropListData]="done"
      class="example-list"
      (cdkDropListDropped)="drop($event)">
      <div class="example-box" *ngFor="let item of done" cdkDrag>{{item}}</div>
    </div>
  </div>
</div>
""")
f.close()

In [ ]:
import IPython
js_code = '''
document.querySelector("#output-area").appendChild(document.createTextNode("hello world!"));
'''
display(IPython.display.Javascript(js_code))

# Mergin 2

In [ ]:
merger = PdfMerger()

merger.append("/content/ALL - ACSR_0821_Prints - Bass.pdf")
merger.append("/content/ALL - ACSR_0821_Prints - Clarinet.pdf")
merger.append("/content/ALL - ACSR_0821_Prints - Drums.pdf")
merger.append("/content/ALL - ACSR_0821_Prints - Electric Guitar.pdf")
merger.append("/content/ALL - ACSR_0821_Prints - Rhythm Guitar.pdf")
merger.append("/content/ALL - ACSR_0821_Prints - Vocals.pdf")
merger.append("/content/ALL - ACSR_0821_Prints - Tenor Saxophone.pdf")

merger.write("ACSR821_Prints.pdf")

merger.close()
